In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
path = "/content/gdrive/MyDrive/MachineLearning/praktikum/prak9/"

In [ ]:
cancer_data = pd.read_csv(path + '/data/data.csv')

In [ ]:
cancer_data.head()

In [ ]:
cancer_data.tail()

rous/kolom

In [ ]:
cancer_data.shape

In [ ]:
cancer_data.info()

Missing Value

In [ ]:
cancer_data.isnull().sum()

Menghapus kolom yang tidak relevan untuk pemodelan

In [ ]:
cancer_data = cancer_data.drop(columns=['Unnamed: 32'], axis=1)

mengecek outliers(data anomali/beda sendiri/gak masuk akal)

In [ ]:
cancer_data.boxplot(column=['radius_mean'])
plt.title("Boxplot of jari-jari sel tumor with Outliers")
plt.show()

In [ ]:
cancer_data.boxplot(column=['area_mean'])
plt.title("Boxplot of luas sel tumor with Outliers")
plt.show()

mengisi missing value pada kolom

In [ ]:
cancer_data['radius_mean'].fillna (cancer_data['radius_mean'].median(), inplace=True)
cancer_data['area_mean'].fillna (cancer_data['area_mean'].median(), inplace=True)

mengecek kembali

In [ ]:
cancer_data.isnull().sum()

Data Analysis dan Encoding Kategorial
statistik deskripttif

In [ ]:
cancer_data.describe()

value counts(meliat banyaknya nilai)

In [ ]:
cancer_data['diagnosis'].value_counts()

In [ ]:
cancer_data['perimeter_mean'].value_counts()

In [ ]:
cancer_data['texture_mean'].value_counts()

Visualisasi data (sns.countplot) menunjukan kolerasi antar fitur

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(20, 15))
axes = axes.flatten()

sns.countplot(x='diagnosis', data=cancer_data, ax=axes[0])
axes[0].set_title('Jumlah Diagnosis (Malignant vs Benign)')

sns.histplot(cancer_data['radius_mean'], kde=True, ax=axes[1])
axes[1].set_title('Distribusi radius_mean')

sns.histplot(cancer_data['area_mean'], kde=True, ax=axes[2])
axes[2].set_title('Distribusi area_mean')

sns.boxplot(x='diagnosis', y='radius_mean', data=cancer_data, ax=axes[3])
axes[3].set_title('radius_mean berdasarkan diagnosis')

sns.boxplot(x='diagnosis', y='area_mean', data=cancer_data, ax=axes[4])
axes[4].set_title('area_mean berdasarkan diagnosis')

sns.heatmap(
    cancer_data[['radius_mean', 'area_mean', 'perimeter_mean', 'texture_mean']].corr(),
    annot=True,
    cmap='coolwarm',
    ax=axes[5]
)
axes[5].set_title('Korelasi radius, area, perimeter, texture')

plt.tight_layout()
plt.show()


Encoding Kategorikal fitur di ubah menjadi nilai numeric agar dapat diproses oleh model

In [ ]:
cancer_data['diagnosis'] = cancer_data['diagnosis'].map({'M': 1, 'B': 0})

In [ ]:
cancer_data.iloc[0:9]

Separating, Splitting, and Scaling Data(independen y, dependen x)

In [ ]:
X = cancer_data.drop(columns=['diagnosis'])
Y = cancer_data['diagnosis']

In [ ]:
cancer_data['diagnosis'].unique()

In [ ]:
X.head()

In [ ]:
Y.head()

Train split (membagi data menjadi 80% untuk training)

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X,
                                                    Y,
                                                    test_size=0.2,
                                                    stratify=Y, # untuk data yang menyimpang
                                                    random_state=42)

In [ ]:
print(X.shape, X_train.shape, X_test.shape)

Melatih scaler pada data ini penting untuk Nauve Bayes

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Naive Bayes Classification
Modeling

In [ ]:
from sklearn.naive_bayes import GaussianNB
nb_model = GaussianNB()
nb_model.fit(X_train_scaled, Y_train)

Accuracy Score & Evaluation

In [ ]:
# Accuracy
train_pred_nb = nb_model.predict(X_train_scaled)
test_pred_nb = nb_model.predict(X_test_scaled)

print("Training Accuracy (NB): ", accuracy_score (Y_train, train_pred_nb))
print("Testing Accuracy (NB): ", accuracy_score (Y_test, test_pred_nb))

Visualisasi Confusion Matrix ( Naive Bayes)

In [ ]:
plt.figure(figsize=(6,4))
cm_nb = confusion_matrix(Y_test, test_pred_nb)

sns.heatmap(cm_nb, annot=True, fmt="d", cmap="Blues",
            xticklabels=['Jinak', 'Ganas'],
            yticklabels=['Jinak', 'Ganas'])

plt.title("Confusion Matrix Naive Bayes")
plt.xlabel("Prediksi")
plt.ylabel("Aktual")
plt.show()


Classification Report

In [ ]:
print("\nClassification Report (NB):")
print(classification_report(Y_test, test_pred_nb))

Cross Validation

In [ ]:
from sklearn.model_selection import cross_val_score
cv_nb = cross_val_score(nb_model, X, Y, cv = 5 , scoring='accuracy')

print("\nNaive Bayes Cross Validation Accuracy (5-Fold):")
print("Scores:", cv_nb)
print("Mean Accuracy:", cv_nb.mean())
print("Std Deviation:", cv_nb.std())